# Imports

In [1]:
import torch
from tokenizers import SentencePieceBPETokenizer
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors, normalizers
from transformers import BitsAndBytesConfig, AddedToken, AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerFast, T5Tokenizer
from peft import LoraConfig, get_peft_model
import sentencepiece as spm

from collections import Counter
from dotenv import load_dotenv
import os
import re

In [2]:
load_dotenv("../.env")

True

# Gemma3

## Tokenizer

In [4]:
MODEL_FILE = "../models/tokenizer/bel_tokenizer.model"

In [5]:
MODEL_NAME = "google/gemma-3-1b-it"

In [6]:
sp = spm.SentencePieceProcessor(model_file=MODEL_FILE)
print(sp.encode("Hello world!", out_type=int))

[738, 2023, 31914, 24022, 31955]


In [7]:
# tokenizer = SentencePieceBPETokenizer("../models/tokenizer/bel_tokenizer.vocab", MODEL_FILE)
# tokenizer.save("../models/tokenizer/hf_bel_tokenizer.json")

In [8]:
tokenizer = T5Tokenizer(
    vocab_file=MODEL_FILE,
    # tokenizer_object=sp,
    pad_token="[PAD]",
    unk_token="[UNK]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]",
    bos_token="<s>",
    eos_token="</s>"
)
tokenizer.save_pretrained("../models/tokenizer/hf_bel_tokenizer")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


('../models/tokenizer/hf_bel_tokenizer/tokenizer_config.json',
 '../models/tokenizer/hf_bel_tokenizer/special_tokens_map.json',
 '../models/tokenizer/hf_bel_tokenizer/spiece.model',
 '../models/tokenizer/hf_bel_tokenizer/added_tokens.json')

In [21]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=os.getenv("HF_TOKEN"))

inputs = tokenizer("Вітам!", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50)

In [24]:
tokenizer

T5Tokenizer(name_or_path='', vocab_size=32000, model_max_length=1000000000000000019884624838656, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]', 'additional_special_tokens': ['<extra_id_0>', '<extra_id_1>', '<extra_id_2>', '<extra_id_3>', '<extra_id_4>', '<extra_id_5>', '<extra_id_6>', '<extra_id_7>', '<extra_id_8>', '<extra_id_9>', '<extra_id_10>', '<extra_id_11>', '<extra_id_12>', '<extra_id_13>', '<extra_id_14>', '<extra_id_15>', '<extra_id_16>', '<extra_id_17>', '<extra_id_18>', '<extra_id_19>', '<extra_id_20>', '<extra_id_21>', '<extra_id_22>', '<extra_id_23>', '<extra_id_24>', '<extra_id_25>', '<extra_id_26>', '<extra_id_27>', '<extra_id_28>', '<extra_id_29>', '<extra_id_30>', '<extra_id_31>', '<extra_id_32>', '<extra_id_33>', '<extra_id_34>', '<extra_id_35>', '<extra_id_36>', '<extra_id_37>', '<extr

In [19]:
outputs[0]

tensor([   100,     14,    485,   7668,  19391,  31955,      2,  19174,   1688,
          3117,    718,   3001, 236787,   1192,    905,    107,  19174,   1688,
          3117,    718,   3001, 236787,   9503,    905,    107,  19174,   1688,
          3117,    718,   3001, 236787,  10671,    905,    107,  19174,   1688,
          3117,    718,   3001, 236787,  38852,    905,    107,  19174,   1688,
          3117,    718,   3001, 236787,  28975,    905,    107,  19174,   1688,
          3117,    718,   3001])

In [20]:
tokenizer.decode(outputs[0], skip_special_tokens=True)

IndexError: piece id is out of range.

## Config

In [2]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

## Fine-tuning

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Qwen

In [3]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

In [4]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


## Example

In [6]:
prompt = "Best tea for an evening"
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [7]:
print(response)

Choosing the best tea for an evening can depend on your personal preferences, but here are some popular options that are both relaxing and enjoyable:

1. **Herbal Teas**:
   - **Chamomile**: Known for its calming effects, perfect for winding down before bed.
   - **Peppermint**: Refreshing and soothing, great for digestion and relaxation.
   - **Lavender**: Calming and helps with stress relief.

2. **Black Tea**:
   - **Earl Grey**: A strong, robust flavor with a hint of bergamot, good for a cozy evening.
   - **English Breakfast**: Rich and full-bodied, ideal for a hearty evening.

3. **Green Tea**:
   - **Matcha**: High in antioxidants and can be enjoyed as a light evening drink.
   - **Sencha**: Light and refreshing, with a subtle sweetness.

4. **White Tea**:
   - **Silver Needle**: Delicate and mild, great for a lighter evening tea.

5. **Oolong Tea**:
   - **Tie Guan Yin**: A balanced and aromatic choice that can be enjoyed in the evening.

6. **Spiced Teas**:
   - **Spiced Chai*

## Check Current tokenizer

In [5]:
text1 = "Переведи на беларуский язык: Чайник с чаем стоит на столе."
text2 = "што гэта за мова?"
text3 = "Вітам!"

In [6]:
def print_id_token(outputs, tokenizer):
    print(f"{'token':<8} -> id")
    for output in outputs:
        for token_id, token in zip(output.to("cpu"), tokenizer.batch_decode(output), strict=True):
            print(f"{token_id.item():<8} -> '{token}'")

In [7]:
def tokenize_text(prompt: str, tokenizer):
    torch.cuda.empty_cache()
    messages = [
        {"role": "system", "content": "You are belarusian teacher. Answer ONLY in belarusian."},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        temperature=0.1,
        top_p=0.9,
        max_new_tokens=512,
    )

    for i, output in enumerate(tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)):
        print(f"Result {i + 1}:\n\t {output.strip()}")
    print_id_token(outputs, tokenizer)

In [8]:
def tokenize_texts(texts: list[str], tokenizer):
    for text in texts:
        print(f"\t\t{text}")
        tokenize_text(text, tokenizer)

In [9]:
tokenize_texts([text1, text2, text3], tokenizer)

		Переведи на беларуский язык: Чайник с чаем стоит на столе.
Result 1:
	 Чайнік з чаем стоіць на столе.
token    -> id
151644   -> '<|im_start|>'
8948     -> 'system'
198      -> '
'
2610     -> 'You'
525      -> ' are'
1656     -> ' bel'
50319    -> 'arus'
1103     -> 'ian'
11079    -> ' teacher'
13       -> '.'
21806    -> ' Answer'
26687    -> ' ONLY'
304      -> ' in'
1656     -> ' bel'
50319    -> 'arus'
1103     -> 'ian'
13       -> '.'
151645   -> '<|im_end|>'
198      -> '
'
151644   -> '<|im_start|>'
872      -> 'user'
198      -> '
'
98574    -> 'Пер'
32642    -> 'ев'
13103    -> 'ед'
1802     -> 'и'
13073    -> ' на'
131395   -> ' бел'
17273    -> 'ар'
43055    -> 'ус'
130366   -> 'кий'
130844   -> ' язык'
25       -> ':'
124713   -> ' Ч'
18530    -> 'ай'
69859    -> 'ник'
5409     -> ' с'
17307    -> ' ч'
30962    -> 'аем'
130960   -> ' стоит'
13073    -> ' на'
130297   -> ' стол'
1504     -> 'е'
13       -> '.'
151645   -> '<|im_end|>'
198      -> '
'
151644   -> '<|im_sta

## Train tokenizer

In [10]:
belarusian_text = "data/processed/qwen_belarusian_v2.txt"

In [11]:
new_tokenizer_dir = "../models/tokenizer/qwen2.5_tokenizer_bel"

In [12]:
CYR = re.compile(r"[АаБбВвГгДдЕеЖжЗзИиЙйКкЛлМмНнОоПпРрСсТтУуФфХхЦцЧчШшЩщЪъЫыЬьЭэЮюЯяўЎіІёЁ]{2,8}")

In [13]:
def batch_iterator(stop: int | None = None):
    i = 0
    with open("../" + belarusian_text, "r", encoding="utf-8") as f:
        for line in f:
            if stop is not None and i >= stop:
                break
            if line == "":
                continue
            i += 1
            if line != "":
                yield line.replace('“', '"').replace('”', '"').replace('‘', "'").replace('’', "'")

In [14]:
new_tokenizer = AutoTokenizer.from_pretrained(model_name)

In [15]:
def yield_ngrams(stop: int | None = None):
    for line in batch_iterator(stop):
        for m in CYR.findall(line):
            yield m

In [16]:
c = Counter(yield_ngrams())
new_tokens = [w for w, _ in c.most_common(50_000) if w not in new_tokenizer.get_vocab()]
print("tokens to add:", len(new_tokens))

tokens to add: 50000


In [17]:
new_tokenizer.add_tokens([AddedToken(t, lstrip=False, rstrip=False) for t in new_tokens])
print("new vocab size:", len(new_tokenizer))
new_tokenizer.save_pretrained(new_tokenizer_dir)

new vocab size: 201665


('qwen2.5_tokenizer_bel/tokenizer_config.json',
 'qwen2.5_tokenizer_bel/special_tokens_map.json',
 'qwen2.5_tokenizer_bel/chat_template.jinja',
 'qwen2.5_tokenizer_bel/vocab.json',
 'qwen2.5_tokenizer_bel/merges.txt',
 'qwen2.5_tokenizer_bel/added_tokens.json',
 'qwen2.5_tokenizer_bel/tokenizer.json')

## Compare Tokenizers old vs new

In [18]:
len(new_tokenizer)

201665

In [26]:
print("\t".join(f"{word} {count:,}" for word, count in sorted(new_tokenizer.get_vocab().items(), key=lambda x: -x[1])[:50]))

Міканор 201,664	дзячнасц 201,663	Ейшиш 201,662	НИОР 201,661	рковъ 201,660	предикат 201,659	СОГЛАСНЫ 201,658	Мігулеўс 201,657	нацгонал 201,656	Роном 201,655	Лунёва 201,654	Крака 201,653	Мондрыха 201,652	радзюжка 201,651	перцем 201,650	кажды 201,649	федэраты 201,648	мула 201,647	маркс 201,646	класц 201,645	тэлеф 201,644	захова 201,643	Макаёнка 201,642	брычку 201,641	будуть 201,640	дэфармац 201,639	Супрун 201,638	спяшалас 201,637	арыштова 201,636	бязмежжа 201,635	зморшчыл 201,634	прысыпан 201,633	паганца 201,632	признает 201,631	сыходз 201,630	паперадж 201,629	понимали 201,628	податей 201,627	земельно 201,626	шэдэўр 201,625	бут 201,624	Старина 201,623	Иордан 201,622	переполн 201,621	сапог 201,620	немецкую 201,619	разъясни 201,618	рацию 201,617	Народной 201,616	томах 201,615


In [27]:
inputs = tokenizer.encode(text1)
new_inputs = new_tokenizer.encode(text1)
len(inputs), len(new_inputs)

(22, 25)

In [28]:
tokenize_texts([text1, text2, text3], new_tokenizer)

		Переведи на беларуский язык: Чайник с чаем стоит на столе.


/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [45,1,0], thread: [96,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [45,1,0], thread: [97,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [45,1,0], thread: [98,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [45,1,0], thread: [99,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [45,1,0], thread: [100,0,0] Asse

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [35]:
for text in [text1, text2, text3]:
    print(f"\t\t\t{text}\n\n\n")
    encoded = tokenizer.encode(text)
    new_encoded = new_tokenizer.encode(text)
    print(f"{encoded=}\n")
    print(f"{new_encoded=}\n")
    print(f"per_token={[tokenizer.decode(token) for token in encoded]}\n")
    print(f"new per_token = {[new_tokenizer.decode(token) for token in new_encoded]}")

			Переведи на беларуский язык: Чайник с чаем стоит на столе.



encoded=[98574, 32642, 13103, 1802, 13073, 131395, 17273, 43055, 130366, 130844, 25, 124713, 18530, 69859, 5409, 17307, 30962, 130960, 13073, 130297, 1504, 13]

new_encoded=[163708, 153657, 196285, 220, 151665, 220, 151701, 151911, 220, 152795, 25, 220, 167838, 165819, 155957, 5409, 220, 190899, 220, 159109, 220, 151665, 220, 186435, 13]

per_token=['Пер', 'ев', 'ед', 'и', ' на', ' бел', 'ар', 'ус', 'кий', ' язык', ':', ' Ч', 'ай', 'ник', ' с', ' ч', 'аем', ' стоит', ' на', ' стол', 'е', '.']

new per_token = ['Пер', 'ев', 'еди', ' ', 'на', ' ', 'беларуск', 'ий', ' ', 'язык', ':', ' ', 'Ча', 'йн', 'ик', ' с', ' ', 'чаем', ' ', 'стоит', ' ', 'на', ' ', 'столе', '.']
			што гэта за мова?



encoded=[12150, 24634, 24725, 67406, 43513, 43846, 11310, 125536, 30]

new_encoded=[151667, 220, 151678, 220, 151670, 220, 152188, 30]

per_token=['ш', 'то', ' г', 'э', 'та', ' за', ' м', 'ова', '?']

new per_token = ['што', ' ', 'гэта',